<h1><center><strong></strong></center></h1>
<h1><center><strong>CME466</strong></center></h1>
<h1><center><strong>Design of an Advanced Digital System</strong></center></h1>
<p><center><strong>Department of Electrical and Computer Engineering</strong></center></p>
<p><center><strong>2025 Winter Term</strong></center></p>
<p><center><strong>Last update: March 08,2026</strong></center></p>

# Convolutional Neural Network (CNN) Part 2
# Deploy on RPi

Food-101 and LeNet-5



# 0. Prerequisites

## 0.1 Import Libraries

In [ ]:
# Import PyTorch
import torch
from torch import nn

# Import torchvision
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor

# Import matplotlib for visualization
import matplotlib.pyplot as plt
from pathlib import Path

# Check versions
# print(f"PyTorch version: {torch.__version__}\ntorchvision version: {torchvision.__version__}")

In [ ]:
# Try to import torchinfo, install if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] coudln't find torchinfo, installing it...")
    !pip install -q torchinfo
    from torchinfo import summary
    print("[INFO] Done!")

In [ ]:
# Try to import torchmetrics and install if it doesn't work
try:
    import torchmetrics
except:
    print("[INFO] coudln't find torchmetrics, installing it...")
    !pip install -q torchmetrics
    import torchmetrics
    print("[INFO] Done!")

## 0.2 Setup Device agnostic code

In [ ]:
# Set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
%matplotlib inline

# 1. Getting a dataset

## 1.1 Custom Dataset - Food-101

<p>The data we're going to be using is a subset of the <a href="https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/">Food-101 dataset</a>.</p>
<p>Food101 is popular computer vision benchmark as it contains 1000 images of 101 different kinds of foods, totaling 101,000 images (75,750 train and 25,250 test).</p>

![Food101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/static/img/food-101.jpg)


<p>We have images of pizza, steak and sushi in standard image classification format.</p><p>Image classification format contains separate classes of images in separate directories titled with a particular class name.</p>
<p>For example, all images of <code>pizza</code> are contained in the <code>pizza/</code> directory.</p>
<p>This format is popular across many different image classification benchmarks, including <a href="https://www.image-net.org/">ImageNet</a> (of the most popular computer vision benchmark datasets).</p>
<p>You can see an example of the storage format below, the images numbers are arbitrary.</p>
<pre><code>pizza_steak_sushi/ &lt;- overall dataset folder
    train/ &lt;- training images
        pizza/ &lt;- class name as folder name
            image01.jpeg
            image02.jpeg
            ...
        steak/
            image24.jpeg
            image25.jpeg
            ...
        sushi/
            image37.jpeg
            ...
    test/ &lt;- testing images
        pizza/
            image101.jpeg
            image102.jpeg
            ...
        steak/
            image154.jpeg
            image155.jpeg
            ...
        sushi/
            image167.jpeg
            ...
</code></pre>

<p>The goal will be to <strong>take this data storage structure and turn it into a dataset usable with PyTorch</strong>.</p>
<blockquote>
<p><strong>Note:</strong> The structure of the data you work with will vary depending on the problem you're working on. But the premise still remains: Find a way to best turn it into a dataset compatible with PyTorch.</p>
</blockquote>

## 1.2 Download the dataset

### Download the dataset directly into colab
We are using a subset of Food-101 dataset that contains only 5 classes.

In [ ]:
!gdown --id 1ARrYMBASqsPomwD8RsgwyPxzqU09igi6 -O food101_5_cls.zip

### Extracting the dataset

In [ ]:
import requests
import zipfile
from pathlib import Path

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "food101_5_cls"

# If the image folder doesn't exist, create it...
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

# Unzip the dataset
with zipfile.ZipFile("food101_5_cls.zip", "r") as zip_ref:
    print("Unzipping the dataset ...")
    zip_ref.extractall(image_path)

<p>We can inspect what's in our data directory by writing a small helper function to walk through each of the subdirectories and count the files present.</p>
<p>To do so, we'll use Python's in-built <a href="https://docs.python.org/3/library/os.html#os.walk"><code>os.walk()</code></a>.</p>
We can also check the folders on the left.....

In [ ]:
import os
def walk_through_dir(dir_path):
  """
  Walks through dir_path returning its contents.
  Args:
    dir_path (str or pathlib.Path): target directory

  Returns:
    A print out of:
      number of subdiretories in dir_path
      number of images (files) in each subdirectory
      name of each subdirectory
  """
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

In [ ]:
walk_through_dir(image_path)

## 1.3 Display an image

### Setup our training and testing paths

In [ ]:
# Setup train and testing paths
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

In [ ]:
import random
from PIL import Image

# Set seed
#random.seed(42)

# 1. Get all image paths (* means "any combination")
image_path_list = list(image_path.glob("*/*/*.jpg"))

# 2. Get random image path
random_image_path = random.choice(image_path_list)

# 3. Get image class from path name (the image class is the name of the directory where the image is stored)
image_class = random_image_path.parent.stem

# 4. Open image
img = Image.open(random_image_path)

# 5. Print metadata
print(f"Random image path: {random_image_path}")
print(f"Image class: {image_class}")
print(f"Image height: {img.height}")
print(f"Image width: {img.width}")
img

Note that the images are of different dimensions (512 x512. or 512 x 384, so on...)

## 1.4 Transforming data

In [ ]:
import torchvision.transforms as transforms

# Helper functions: Write transform for image
train_transform = transforms.Compose([
    # Resize the images to 64x64
    transforms.Resize(size=(64, 64)),
    # Flip the images randomly on the horizontal
    transforms.RandomHorizontalFlip(p=0.5), # p = probability of flip, 0.5 = 50% chance
    # Flip the images randomly on the vertical
    transforms.RandomVerticalFlip(p=0.5),
    # Rotate images randomly
    transforms.RandomRotation(degrees=30),
    # Turn the image into a torch.Tensor
    transforms.ToTensor() # converts all pixels (H, W, C)[0, 255] to (C, H, W)[0.0, 1.0]
])

test_transform = transforms.Compose([
    # Resize the images to 64x64
    transforms.Resize(size=(64, 64)),
    # Turn the image into a torch.Tensor
    transforms.ToTensor() # converts all pixels (H, W, C)[0, 255] to (C, H, W)[0.0, 1.0]
])

### Visualize transformations

In [ ]:
import matplotlib.pyplot as plt

def plot_transformed_images(image_paths, transform, n=3, seed=40):
    """Plots a series of random images from image_paths.

    Will open n image paths from image_paths, transform them
    with transform and plot them side by side.

    Args:
        image_paths (list): List of target image paths.
        transform (PyTorch Transforms): Transforms to apply to images.
        n (int, optional): Number of images to plot. Defaults to 3.
        seed (int, optional): Random seed for the random generator. Defaults to 42.
    """
    random.seed(seed)
    random_image_paths = random.sample(image_paths, k=n)
    for image_path in random_image_paths:
        with Image.open(image_path) as f:
            fig, ax = plt.subplots(1, 2)
            ax[0].imshow(f)
            ax[0].set_title(f"Original \nSize: {f.size}")
            ax[0].axis("off")

            # Transform and plot image
            # Note: permute() will change shape of image to suit matplotlib
            # (PyTorch default is [C, H, W] but Matplotlib is [H, W, C])
            transformed_image = transform(f).permute(1, 2, 0)
            ax[1].imshow(transformed_image)
            ax[1].set_title(f"Transformed \nSize: {transformed_image.shape}")
            ax[1].axis("off")

            fig.suptitle(f"Class: {image_path.parent.stem}", fontsize=16)

plot_transformed_images(image_path_list,
                        transform=train_transform,
                        n=3)

## 1.5 Loading Image Data into torchvision.datasets.ImageFolder
Using <a href="https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html#torchvision.datasets.ImageFolder"><code>ImageFolder</code>

In [ ]:
# Use ImageFolder to create dataset(s)
from torchvision import datasets
train_data = datasets.ImageFolder(root=train_dir, # target folder of images
                                  transform=train_transform, # transforms to perform on data (images)
                                  target_transform=None) # no transforms on labels

test_data = datasets.ImageFolder(root=test_dir,
                                 transform=test_transform)

print(f"Train data:\n{train_data}\nTest data:\n{test_data}")

In [ ]:
# Get class names as a list
class_names = train_data.classes
class_names

In [ ]:
# Verify the lengths (same as the number of datapoints above)
len(train_data), len(test_data)

## 1.6 Image Shapes


Let's check out the first sample of the training data.

In [ ]:
img, label = train_data[125][0], train_data[125][1]
print(f"Image tensor:\n{img}")
print(f"Image shape: {img.shape}")
print(f"Image datatype: {img.dtype}")
print(f"Image label: {label}")
print(f"Label datatype: {type(label)}")

# 2. Prepare DataLoader

Thanks to the modular and clean code we've written, we don't need to change this part :)

In [ ]:
# Turn train and test Datasets into DataLoaders
from torch.utils.data import DataLoader

train_dataloader = DataLoader(dataset=train_data,
                              batch_size=32, # how many samples per batch?
                              num_workers=1, # how many subprocesses to use for parallel data loading? (higher = more)
                              shuffle=True) # shuffle the data?

test_dataloader = DataLoader(dataset=test_data,
                             batch_size=32,
                             num_workers=1,
                             shuffle=False) # don't shuffle testing data


img, label = next(iter(train_dataloader))

# Batch size will now be 1, try changing the batch_size parameter above and see what happens
print(f"Image shape: {img.shape}") #[batch_size, color_channels, height, width]
print(f"Label shape: {label.shape}")

# 3. Helper Functions

In [ ]:
from torchmetrics.classification import MulticlassAccuracy

In [ ]:
# Move values to device
#torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               num_classes,
               device: torch.device = device):
    """Evaluates a given model on a given dataset.

    Args:
        model (torch.nn.Module): A PyTorch model capable of making predictions on data_loader.
        data_loader (torch.utils.data.DataLoader): The target dataset to predict on.
        loss_fn (torch.nn.Module): The loss function of model.
        num_classes (int): Number of classes in dataset.
        device (str, optional): Target device to compute on. Defaults to device.

    Returns:
        (dict): Results of model making predictions on data_loader.
    """
    acc = MulticlassAccuracy(num_classes=num_classes, average='macro').to(device)
    loss = 0
    model.eval()
    with torch.inference_mode():
        for X, y in data_loader:
            # Send data to the target device
            X, y = X.to(device), y.to(device)
            y_logits = model(X)
            loss += loss_fn(y_logits, y)
            y_pred_probs = torch.softmax(y_logits, dim=1)
            y_pred_labels = torch.argmax(y_pred_probs, dim=1)
            acc.update(y_pred_labels, y)

        # Scale loss and acc
        loss /= len(data_loader)
        test_acc = acc.compute().cpu().numpy()
        acc.reset()

    return {"model_name": model.__class__.__name__, # only works when model was created with a class
            "model_loss": loss.item(),
            "model_acc": test_acc.item()}

In [ ]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               num_classes,
               device: torch.device = device):
    # Instantiate Multiclass Accuracy instance
    acc = MulticlassAccuracy(num_classes=num_classes, average='micro').to(device)
    train_loss = 0
    model.to(device)
    for batch, (X, y) in enumerate(data_loader):
        # Send data to GPU
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_logits = model(X)

        # 2. Calculate loss
        loss = loss_fn(y_logits, y)
        train_loss += loss
        y_pred_prob = torch.softmax(y_logits, dim=1)
        y_pred_labels = torch.argmax(y_pred_prob, dim=1)
        acc.update(y_pred_labels, y)

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

    # Calculate loss and accuracy per epoch and print out what's happening
    train_loss /= len(data_loader)
    train_acc = acc.compute().cpu().numpy()

    acc.reset()

    print(f"Train loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")

    return train_loss, train_acc

In [ ]:
def test_step(data_loader: torch.utils.data.DataLoader,
              model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              num_classes,
              device: torch.device = device):

    acc = MulticlassAccuracy(num_classes=num_classes, average='micro').to(device)
    test_loss = 0
    model.to(device)
    model.eval() # put model in eval mode
    # Turn on inference context manager
    with torch.inference_mode():
        for X, y in data_loader:
            # Send data to GPU
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_logits = model(X)

            # 2. Calculate loss and accuracy
            test_loss += loss_fn(test_logits, y)
            test_pred_probs = torch.softmax(test_logits, dim=1)
            test_pred_labels = torch.argmax(test_pred_probs, dim=1)
            acc.update(test_pred_labels, y)

        # Adjust metrics and print out
        test_loss /= len(data_loader)
        test_acc = acc.compute().cpu().numpy()

        acc.reset()
        print(f"Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%\n")
        return test_loss, test_acc

In [ ]:
import matplotlib.pyplot as plt
def visualize_training(results):

    # Plot loss
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(range(1, len(results["train_loss"]) + 1), results["train_loss"], 'bo-', label='Train Loss')
    plt.plot(range(1, len(results["train_loss"]) + 1), results["test_loss"], 'ro-', label='Test Loss')
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training & Testing Loss")
    plt.legend()
    plt.grid()
    plt.ylim(0, max(max(results["train_loss"]), max(results["test_loss"])) * 1.1)  # Ensure the loss starts from 0

    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(range(1, len(results["train_loss"]) + 1), results["train_acc"], 'bo-', label='Train Accuracy')
    plt.plot(range(1, len(results["train_loss"]) + 1), results["test_acc"], 'ro-', label='Test Accuracy')
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.title("Training & Testing Accuracy")
    plt.legend()
    plt.grid()
    plt.ylim(0, 1)  # Accuracy should be between 0% and 100%

    plt.show()

In [ ]:
# Plot Predictions
import numpy as np

def plot_image(i, predictions_array, true_label, img):
    true_label, img = true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])

    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

    predicted_label = torch.argmax(predictions_array).item()

    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'

    plt.xlabel("{} {:2.0f}% ({})".format(class_names[predicted_label],
                                100*torch.max(predictions_array).item(),
                                class_names[true_label]),
                                color=color)

def plot_value_array(i, predictions_array, true_label, num_classes=10):
  true_label = true_label[i]
  plt.grid(False)
  plt.xticks(range(num_classes))
  plt.yticks([])
  thisplot = plt.bar(range(num_classes), predictions_array, color="#777777")
  plt.ylim([0, 1])
  predicted_label = np.argmax(predictions_array)

  thisplot[predicted_label].set_color('red')
  thisplot[true_label].set_color('blue')

# 4. Model 1: Let us use a basic model (LeNet-5)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Food101ModelV1(nn.Module):
    def __init__(self, num_classes:int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3,
                               out_channels=6,
                               kernel_size=5,
                               stride=1,
                               padding=0)
        self.bn1 = nn.BatchNorm2d(6)  # BatchNorm after first conv layer

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # one pool is good; reused

        self.conv2 = nn.Conv2d(in_channels=6,
                               out_channels=16,
                               kernel_size=5,
                               stride=1,
                               padding=0)
        self.bn2 = nn.BatchNorm2d(16)  # BatchNorm after second conv layer

        self.conv3 = nn.Conv2d(in_channels=16,
                               out_channels=10,
                               kernel_size=5,
                               stride=1,
                               padding=0)
        self.bn3 = nn.BatchNorm2d(10)  # BatchNorm after second conv layer

        self.fc1 = nn.Linear(in_features=10 * 4 * 4, out_features=128)
        self.bn4 = nn.BatchNorm1d(128)  # BatchNorm after first fully connected layer

        self.fc2 = nn.Linear(in_features=128, out_features=num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # Apply BatchNorm after Conv, before ReLU
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)  # flatten all dimensions except batch
        x = F.relu(self.bn4(self.fc1(x)))
        x = self.fc2(x)
        return x


In [ ]:
torch.manual_seed(42)
model_1 = Food101ModelV1(num_classes=len(class_names)).to(device) # send model to GPU if it's available

next(model_1.parameters()).device # check model device

Check total model parameters..

In [ ]:
from torchinfo import summary
summary(model=model_1,
        input_size=(32, 3, 64, 64),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"])

## 4.1 Setup loss, optimizer and evaluation metrics

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_1.parameters(),
                            lr=0.001)

## 4.2 Training the model

In [ ]:
# Import tqdm for progress bar
from tqdm.auto import tqdm
torch.manual_seed(42)

epochs = 10 # takes long time; start with lower number

# Create empty results dictionary
results = {"train_loss": [],
           "train_acc": [],
           "test_loss": [],
           "test_acc": []
}

for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n---------")
    train_loss, train_acc = train_step(data_loader=train_dataloader,
                                       model=model_1,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer,
                                       num_classes=len(class_names)
                                    )
    test_loss, test_acc = test_step(data_loader=test_dataloader,
                                    model=model_1,
                                    loss_fn=loss_fn,
                                    num_classes=len(class_names)
                                )

    # Update results dictionary
    results["train_loss"].append(train_loss.detach().cpu().numpy())
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss.detach().cpu().numpy())
    results["test_acc"].append(test_acc)

## 4.3 Visualizing results

In [ ]:
visualize_training(results)

# 5. Plot predictions

In [ ]:
import random

# Plot the first X test images, their predicted labels, and (true labels).
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 3
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))

# Convert dataloader into a list of batches
all_batches = list(test_dataloader)

# Pick a random batch
X_batch, y_batch = random.choice(all_batches)
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# Get model predictions
model_1.eval()  # Ensure model is in eval mode
with torch.no_grad():
    y_pred = model_1(X_batch)  # Get raw logits
    y_pred_prob = torch.softmax(y_pred, dim=1)  # Convert logits to probabilities
    y_pred_labels = torch.argmax(y_pred_prob, dim=1)  # Get predicted labels

for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, y_pred_prob[i].cpu(), y_batch.cpu(), X_batch.squeeze().cpu())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, y_pred_prob[i].cpu(), y_batch.cpu(), num_classes=len(class_names))
plt.tight_layout()
plt.show()

# 6. Plot Confusion Matrix

In [ ]:
# Try to import torchmetrics and install if it doesn't work
try:
    import torchmetrics
except:
    print("[INFO] coudln't find torchmetrics, installing it...")
    !pip install -q torchmetrics
    import torchmetrics
    print("[INFO] Done!")

In [ ]:
import matplotlib.pyplot as plt
from torchmetrics import ConfusionMatrix
import seaborn as sn
import pandas as pd

# Assuming 'preds' and 'target' are your accumulated tensors of predictions and true labels
# preds should be an integer tensor of predicted labels (e.g., from torch.argmax(outputs, dim=1))
# target should be an integer tensor of true labels

num_classes = 5 # Set the number of classes
metric = ConfusionMatrix(task="multiclass", num_classes=num_classes)

model_1.eval() # Set model to evaluation mode
all_preds = []
all_targets = []
with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        pred_logits = model_1(X)
        pred_labels = pred_logits.argmax(dim=1)
        all_preds.append(pred_labels)
        all_targets.append(y)

# Concatenate all predictions and targets
y_pred = torch.cat(all_preds)
y_true = torch.cat(all_targets)

confmat = ConfusionMatrix(num_classes=num_classes, task='multiclass')
# Ensure tensors are on CPU for torchmetrics if device is CUDA
cm_tensor = confmat(y_pred.cpu(), y_true.cpu())

# Convert to pandas DataFrame for better visualization with seaborn
df_cm = pd.DataFrame(cm_tensor.numpy(), index=range(num_classes), columns=range(num_classes))
plt.figure(figsize = (7,5))
# Use seaborn's heatmap for a nice plot
sn.heatmap(df_cm, annot=True, fmt="d", cmap='Blues')
plt.xlabel("Predicted labels")
plt.ylabel("True labels")
plt.show()

# 7. Deploying the model on RPi

In [ ]:
!pip install onnx

## 7.1 Convert the PyTorch model to ONNX

In [ ]:
import torch
import torch.onnx

model_1.eval()

# Define dummy input (match your input size)
dummy_input = torch.randn(1, 3, 64, 64).to(device)

# Convert to ONNX format
torch.onnx.export(model_1, dummy_input, "model.onnx", export_params=True, opset_version=11,
                  input_names=['input'], output_names=['output'])


# 8. Resources

<li><a href="https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/">Food101</a></li>
<li><a href="https://www.learnpytorch.io/">Learn PyTorch for Deep Learning: Zero to Mastery book</a></li>
<li><a href="https://lightning.ai/docs/torchmetrics/stable/classification/accuracy.html">Accuracy</a></li>
<li><a href="https://pytorch.org/vision/main/generated/torchvision.datasets.CIFAR10.html">CIFAR10</a></li>
<li><a href="https://www.geeksforgeeks.org/ml-introduction-to-transfer-learning/">What is Transfer Learning?</a></li>